# Discussion 1: Pandas Review

This section of Discussion 1 is meant to review [Pandas](https://pandas.pydata.org/docs/), one of the most popular Python libraries for data-wrangling. It's a crucial tool in any machine learning researcher or engineer's repertoire and you will continue to use it heavily throughout the semester.

Let's start by loading the necessary packages for today's exercise, which will look at movie data from IMDb. We use the `read_csv` function to load data from the internet, but you can also use this function to load a file from your local storage.

*NOTE: The output of the cells may not be correct if run out of order, even if your code is correct. If in doubt, you can run all cells at once, which should take no more than a few seconds.*


In [2]:
import os, random, numpy as np
SEED = 189

os.environ["PYTHONHASHSEED"] = str(SEED)

random.seed(SEED)
np.random.seed(SEED)

In [26]:
import pandas as pd
import plotly.express as px

# Load the title_basics dataset from IMDb
title_basics  = pd.read_csv(
    "https://datasets.imdbws.com/title.basics.tsv.gz",
    sep="\t", compression="gzip", na_values="\\N", nrows=500000
)
# Load the title_ratings dataset from IMDb
title_ratings = pd.read_csv(
    "https://datasets.imdbws.com/title.ratings.tsv.gz",
    sep="\t", compression="gzip", na_values="\\N", nrows=500000
)

# Sort both dataframes by 'tconst' and reset the index
title_basics  = title_basics.sort_values("tconst", kind="mergesort").reset_index(drop=True)
title_ratings = title_ratings.sort_values("tconst", kind="mergesort").reset_index(drop=True)

## Part 1: Exploration and Data Cleaning

Let's start by inspecting the `title_basics` `DataFrame`.

### Q 1.1

How many columns are in the `title_basics` `DataFrame`?
9

What is the data type of the startYear column? Does this make sense?
float64


In [27]:
# YOUR CODE HERE
print(title_basics.shape[1])
print(title_basics["startYear"].dtype)


9
float64


### Q1.2

What is the value in 101st row of the `primaryTitle` column of the `title_basics` `DataFrame`? *HINT: Recall that* `DataFrame` *uses 0-indexing*

"Smarter than the Teacher"


In [28]:

# YOUR CODE HERE
print(title_basics.loc[[101], ["primaryTitle"]])


                 primaryTitle
101  Smarter than the Teacher


### Q1.3

Display the first 3 rows and the last 6 rows of the `title_basics` `DataFrame` as a single `DataFrame`.


In [29]:

# YOUR CODE HERE
print(pd.concat([title_basics.head(3), title_basics.tail(6)], ignore_index=True))


      tconst  titleType                                       primaryTitle  \
0  tt0000001      short                                         Carmencita   
1  tt0000002      short                             Le clown et ses chiens   
2  tt0000003      short                                       Poor Pierrot   
3  tt0520741  tvEpisode  Jörg Kachelmann, Claudia Kleinert, Daniel Kübl...   
4  tt0520742  tvEpisode                      Episode dated 9 February 2004   
5  tt0520743  tvEpisode                        Episode dated 29 March 2004   
6  tt0520744  tvEpisode                        Episode dated 19 April 2004   
7  tt0520745  tvEpisode                          Episode dated 24 May 2004   
8  tt0520746  tvEpisode                    Episode dated 27 September 2004   

                                       originalTitle  isAdult  startYear  \
0                                         Carmencita        0     1894.0   
1                             Le clown et ses chiens        0     1

### Q1.4

How many unique `titleTypes` are there in the `title_basics` `DataFrame`? Which is the most common?


In [30]:

# YOUR CODE HERE
print(title_basics["titleType"].unique().shape[0])

10


Now let's practice some common `DataFrame` modifications.

### Q1.5

Remove the `originalTitle` and `endYear` columns from the `title_basics` `DataFrame`. Make sure that the columns are permanently removed from the `title_basics` `DataFrame`.


In [31]:

# YOUR CODE HERE
title_basics = title_basics.drop(columns=["originalTitle", "endYear"])
title_basics.head()

,tconst,titleType,primaryTitle,isAdult,startYear,runtimeMinutes,genres
0,tt0000001,short,Carmencita,0,1894.0,1.0,"Documentary,Short"
1,tt0000002,short,Le clown et ses chiens,0,1892.0,5.0,"Animation,Short"
2,tt0000003,short,Poor Pierrot,0,1892.0,5.0,"Animation,Comedy,Romance"
3,tt0000004,short,Un bon bock,0,1892.0,12.0,"Animation,Short"
4,tt0000005,short,Blacksmith Scene,0,1893.0,1.0,Short


### Q1.6

Rename `primaryTitle` to `title` and `startYear` to `year` in the `title_basics` `DataFrame`. Make sure that the changes are reflected permanently in the `title_basics` `DataFrame`.



In [32]:

title_basics = title_basics.rename(columns={"primaryTitle": "title", "startYear": "year"})

title_basics.head()

,tconst,titleType,title,isAdult,year,runtimeMinutes,genres
0,tt0000001,short,Carmencita,0,1894.0,1.0,"Documentary,Short"
1,tt0000002,short,Le clown et ses chiens,0,1892.0,5.0,"Animation,Short"
2,tt0000003,short,Poor Pierrot,0,1892.0,5.0,"Animation,Comedy,Romance"
3,tt0000004,short,Un bon bock,0,1892.0,12.0,"Animation,Short"
4,tt0000005,short,Blacksmith Scene,0,1893.0,1.0,Short


### Q1.7

A crucial step in most data processing pipelines for machine learning is dealing with missing or corrupted data. Often, these missing values are represented as a `NaN` (not a number).

Sometimes in the context of machine learning we'd want to estimate a value for a missing feature rather than remove that sample point entirely. Can you think of some simple ways in which we could perform that estimation?

线性插值, 固定填充，平均值、中位数。。。

### Q1.8

Remove all rows from the `title_basics` `DataFrame` where `runtimeMinutes` or `year` is `NaN`.

In [35]:

initial_length = title_basics.shape[0]

title_basics = title_basics.dropna(subset=["runtimeMinutes", "year"])

final_length = title_basics.shape[0]

print(f"{initial_length - final_length} rows removed from dataframe")


144379 rows removed from dataframe


### Q1.9

Change the data type of the `year` column in the `title_basics` `DataFrame` to something that makes more sense. Then confirm that the change is permanently applied.

In [38]:

# YOUR CODE HERE
title_basics["year"] = title_basics["year"].astype(int)
title_basics["year"].dtype


dtype('int64')

Let's practice some more basic filtering and sorting now.

### Q1.10

Extract the feature films (`titleType == "movie"`) released in 1954 from the `title_basics` `DataFrame` (save this as a new `DataFrame`, `feature_films_1954`).

In [40]:
# YOUR CODE HERE
feature_films_1954 = title_basics[
    (title_basics["titleType"] == "movie") & (title_basics["year"] == 1954)
]
feature_films_1954.head()

,tconst,titleType,title,isAdult,year,runtimeMinutes,genres
35844,tt0036493,movie,Mystery of the Black Jungle,0,1954,80.0,"Action,Adventure,Mystery"
36914,tt0037585,movie,Knights of the Queen,0,1954,79.0,Adventure
37343,tt0038020,movie,Relato policíaco,0,1954,75.0,Crime
37410,tt0038089,movie,Siluri umani,0,1954,87.0,"Drama,War"
37558,tt0038240,movie,Das Licht der Liebe,0,1954,95.0,Drama


### Q1.11

Among the feature films from 1954, which film has the longest runtime? Return its `title` and `runtimeMinutes` as a `DataFrame` extracted from the `feature_films_1954` `DataFrame`.


In [52]:
# YOUR CODE HERE
longest_feature_films_1954 = feature_films_1954.sort_values("runtimeMinutes", ascending=False).head(1)[["title", "runtimeMinutes"]]
longest_feature_films_1954


,title,runtimeMinutes
46203,Gunfighters of the Northwest,315.0


## Part 2: Complex Modifications, Aggregations, Merging, and Plotting

Let's first modify the `title_basics` `DataFrame` so that we have one genre in for each row by duplicating rows that have multiple genres. [`df.explode`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.explode.html) will be helpful for this transformation.

In [53]:
# Split the genres string into a list of genres
title_basics['genres'] = title_basics['genres'].str.split(',')
# Explode the list of genres into separate rows
title_basics = title_basics.explode('genres')

title_basics.head()

,tconst,titleType,title,isAdult,year,runtimeMinutes,genres
0,tt0000001,short,Carmencita,0,1894,1.0,Documentary
0,tt0000001,short,Carmencita,0,1894,1.0,Short
1,tt0000002,short,Le clown et ses chiens,0,1892,5.0,Animation
1,tt0000002,short,Le clown et ses chiens,0,1892,5.0,Short
2,tt0000003,short,Poor Pierrot,0,1892,5.0,Animation


### Q2.1

For each `genre` in the `title_basics` `DataFrame`, compute the mean runtime of feature films released since 1960.
Show the five longest-mean genres.

In [64]:

# YOUR CODE HERE
q21 = (title_basics[
    (title_basics["titleType"] == "movie") & (title_basics["year"] >= 1960)
    ]
    .groupby("genres")["runtimeMinutes"].mean()
    .sort_values(ascending=False)
    .head(5)
    )

q21



genres
History      111.918998
Musical      107.464169
Biography    103.636036
War          102.734879
Romance      102.309448
Name: runtimeMinutes, dtype: float64

### Q2.2

Merge the `title_ratings` `DataFrame` with the `title_basics` `DataFrame` by joining on the `tconst` column. How many titles are present in the `title_basics` `DataFrame` but not in the `title_ratings` `DataFrame`? Store the merged `DataFrame` as `merged_df`.

**Hint:** Recall that because of the genre splitting, the number of titles is not equal to the number of rows.

In [72]:
n_titles_basics = title_basics.groupby("tconst").any().shape[0]

merged_df = pd.merge(title_basics, title_ratings, on="tconst", how="inner")
print(merged_df.head())

n_titles_merged = merged_df.groupby("tconst").any().shape[0]

print(f"\nNumber of titles in basics but not in ratings: {n_titles_basics - n_titles_merged}")

      tconst titleType                   title  isAdult  year  runtimeMinutes  \
0  tt0000001     short              Carmencita        0  1894             1.0   
1  tt0000001     short              Carmencita        0  1894             1.0   
2  tt0000002     short  Le clown et ses chiens        0  1892             5.0   
3  tt0000002     short  Le clown et ses chiens        0  1892             5.0   
4  tt0000003     short            Poor Pierrot        0  1892             5.0   

        genres  meanRuntimeByGenre  averageRating  numVotes  
0  Documentary                 NaN            5.7      2201  
1        Short                 NaN            5.7      2201  
2    Animation                 NaN            5.5       312  
3        Short                 NaN            5.5       312  
4    Animation                 NaN            6.4      2314  

Number of titles in basics but not in ratings: 111102


### Q2.3

Using the `merged_df` `DataFrame` and plotly express, create an interactive scatter plot of the `runtimeMinutes` vs. `numVotes` for movies in the `merged_df` `DataFrame`.
Color the points by the `year` of the movie and add a title and axis labels to the plot. Also, make sure the movie title is visible when hovering over the
data points.

**Note:** To make the data easier to visualize, we take a sample of just 2000 movies. That's why you may not see your favorites on this plot. It's important not to change the random state as you'll end up getting different results for the following questions.

In [74]:
sampled_df = merged_df[merged_df['titleType'] == 'movie'].sample(n=2000, random_state=SEED)

# YOUR CODE HERE
px.scatter(sampled_df, x="runtimeMinutes", y="numVotes", hover_data=["title"], color="year")


ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'customdata': array([['Colorado'],
                                   ['Red Shadow'],
                                   ['Half Baked'],
                                   ...,
                                   ['Sicko'],
                                   ['American Virgin'],
                                   ['The Blue Iguana']], shape=(2000, 1), dtype=object),
              'hovertemplate': ('runtimeMinutes=%{x}<br>numVote' ... '%{marker.color}<extra></extra>'),
              'legendgroup': '',
              'marker': {'color': {'bdata': ('lAfRB84HngfEB9MHrQesB9UHygfLB8' ... 'eTB4kHwAetB6oHrQfXB9cHzwfEBw=='),
                                   'dtype': 'i2'},
                         'coloraxis': 'coloraxis',
                         'symbol': 'circle'},
              'mode': 'markers',
              'name': '',
              'showlegend': False,
              'type': 'scattergl',
              'x': {'bdata': ('AAAAAACATEAAAAAAAABbQAAAAAAAgF' ... 'AAwF5AAAAAAAAAVkAAAAAAAIBWQA=='),
                    'dtype': 'f8'},
              'xaxis': 'x',
              'y': {'bdata': ('QgEAANoDAADqFAEAOgEAAAgAAAAkAA' ... 'AAggAAAA0BAAC0MQEAxwYAAIkCAAA='),
                    'dtype': 'i4'},
              'yaxis': 'y'}],
    'layout': {'coloraxis': {'colorbar': {'title': {'text': 'year'}},
                             'colorscale': [[0.0, '#0d0887'], [0.1111111111111111,
                                            '#46039f'], [0.2222222222222222,
                                            '#7201a8'], [0.3333333333333333,
                                            '#9c179e'], [0.4444444444444444,
                                            '#bd3786'], [0.5555555555555556,
                                            '#d8576b'], [0.6666666666666666,
                                            '#ed7953'], [0.7777777777777778,
                                            '#fb9f3a'], [0.8888888888888888,
                                            '#fdca26'], [1.0, '#f0f921']]},
               'legend': {'tracegroupgap': 0},
               'margin': {'t': 60},
               'template': '...',
               'xaxis': {'anchor': 'y', 'domain': [0.0, 1.0], 'title': {'text': 'runtimeMinutes'}},
               'yaxis': {'anchor': 'x', 'domain': [0.0, 1.0], 'title': {'text': 'numVotes'}}}
})

### Q2.4

Describe any trends you see in the plot above.

越新的电影票数越多

### Q2.5
Which two movies in the plot received the most votes and had the longest runtime, respectively? When were they each released?

tangled, War of the World

# Part 3: Finding the perfect movie

Aakarsh has spent his whole summer brainrotting and doomscrolling, so now his attention span is COOKED. He wants to pick a movie to watch tonight but wants to make sure it isn't so long he gets bored. He decides to construct a Brainrot Score (BRS) to help him find the perfect movie:

$$BRS = \frac{\text{averageRating}}{\sqrt{\text{runtimeMinutes}}}$$

He also wants to make sure the following criteria hold:
- The title should be a *movie* made in 1980 or later.
- It must have at least 10000 votes.
- It must be in the `History`, `Thriller`, or `Comedy` genres.

Can you help Aakarsh out by finding the 3 best movies by BRS in each of his preferred genres?

In [80]:
# YOUR CODE HERE
candidate = (merged_df[
    (merged_df["titleType"] == "movie") &
    (merged_df["year"] >= 1980) & 
    (merged_df["numVotes"] >= 100000) &
    (merged_df["genres"].isin(["History", "Thriller", "Comedy"]))]
    )      
candidate["BRS"] = candidate["averageRating"] / np.sqrt(candidate["runtimeMinutes"])
print(candidate.sort_values(by="BRS", ascending=False).groupby("genres").head(3))

           tconst titleType                       title  isAdult  year  \
165821  tt0114709     movie                   Toy Story        0  1995   
206439  tt0154506     movie                   Following        0  1998   
137709  tt0096283     movie          My Neighbor Totoro        0  1988   
124914  tt0088258     movie          This Is Spinal Tap        0  1984   
184281  tt0130827     movie                Run Lola Run        0  1998   
151698  tt0105236     movie              Reservoir Dogs        0  1992   
398651  tt0433383     movie  Good Night, and Good Luck.        0  2005   
375337  tt0395169     movie                Hotel Rwanda        0  2004   
422049  tt0475276     movie                   United 93        0  2006   

        runtimeMinutes    genres  meanRuntimeByGenre  averageRating  numVotes  \
165821            81.0    Comedy                 NaN            8.3   1169546   
206439            69.0  Thriller                 NaN            7.4    110000   
137709          